# Índice de Caminabilidad — Piloto Av. Roosevelt
**ITT Cali Inteligente · Equipo de Gobierno de Datos**

Este notebook calcula las métricas de caminabilidad para el área de influencia de la intervención en Av. Roosevelt, usando la red peatonal de OpenStreetMap como línea base.

**Instrucciones:**
1. Ejecutar las celdas en orden (Shift + Enter)
2. El GeoJSON se carga automáticamente desde `data/itt_roosevelt/Roosevelt/Geojson_Roosevelt/`
3. Los resultados y el mapa se generan automáticamente

In [ ]:
# Celda 1 — Instalar dependencias
!pip install osmnx geopandas -q

In [ ]:
# Celda 2 — Ruta al GeoJSON del polígono de intervención
# Ruta relativa desde notebooks/ hacia data/itt_roosevelt/Roosevelt/Geojson_Roosevelt/
GEOJSON_PATH = '../data/itt_roosevelt/Roosevelt/Geojson_Roosevelt/Geojson_tramos_Roosevelt_Buffer_100.geojson'

# Si se ejecuta en Google Colab, descomentar estas líneas:
# from google.colab import files
# uploaded = files.upload()
# GEOJSON_PATH = 'Geojson_tramos_Roosevelt_Buffer_100.geojson'

In [ ]:
# Celda 3 — Cargar polígono y calcular área
import geopandas as gpd
import osmnx as ox
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Cargar polígono de intervención
gdf = gpd.read_file(GEOJSON_PATH)
polygon = gdf.geometry.iloc[0]

# Calcular área en sistema métrico (EPSG:3116 — Colombia)
gdf_m = gdf.to_crs(epsg=3116)
area_m2 = gdf_m.geometry.area.iloc[0]

print('=== ÁREA DE INTERVENCIÓN ===')
print(f'  {area_m2:,.0f} m²')
print(f'  {area_m2/10000:.2f} hectáreas')
print(f'  Buffer aplicado: {gdf["BUFF_DIST"].iloc[0]:.0f} m a cada lado del eje')
print(f'  Longitud aproximada del corredor: {gdf["Shape_Leng"].iloc[0]:.0f} m')

In [ ]:
# Celda 4 — Descargar red peatonal desde OpenStreetMap
print('Descargando red peatonal desde OpenStreetMap...')
G = ox.graph_from_polygon(polygon, network_type='walk')
print(f'Red descargada: {len(G.nodes)} nodos, {len(G.edges)} segmentos')

In [ ]:
# Celda 5 — Calcular métricas de caminabilidad
# Pasar área en m² para que OSMnx calcule densidades correctamente
stats = ox.basic_stats(G, area=area_m2)

# Calcular densidad manualmente como respaldo si la versión no la incluye
longitud_total_km = stats['edge_length_total'] / 1000
area_km2 = area_m2 / 1_000_000
densidad_km_km2 = longitud_total_km / area_km2

resultados = {
    'Intersecciones peatonales':       stats['intersection_count'],
    'Longitud red peatonal (km)':      round(longitud_total_km, 2),
    'Longitud promedio segmento (m)':  round(stats['edge_length_avg'], 1),
    'Densidad calle (km/km²)':         round(densidad_km_km2, 2),
}

print('=== MÉTRICAS DE CAMINABILIDAD — LÍNEA BASE ===')
print(f'  Fecha de referencia: OSM actual')
print()
for k, v in resultados.items():
    print(f'  {k}: {v}')

print()
print('Nota: guardar estos valores como línea base para comparar')
print('después de ejecutadas las obras de intervención.')

In [ ]:
# Celda 6 — Mapa de la red peatonal
import matplotlib.pyplot as plt

fig, ax = ox.plot_graph(
    G,
    node_size=12,
    edge_linewidth=1.2,
    bgcolor='white',
    node_color='#2A9C8A',
    edge_color='#1A3A4A',
    figsize=(12, 12),
    show=False,
    close=False
)

ax.set_title(
    'Red peatonal — Área de influencia Av. Roosevelt\nLínea base ITT · Cali Inteligente',
    fontsize=13, pad=15
)

plt.tight_layout()
plt.savefig('../outputs/figures/roosevelt_red_peatonal_linea_base.png', dpi=150, bbox_inches='tight')
plt.show()
print('Mapa guardado en: outputs/figures/roosevelt_red_peatonal_linea_base.png')

In [ ]:
# Celda 7 — Exportar resultados a CSV
import datetime

df_resultados = pd.DataFrame([
    {
        'fecha_medicion': datetime.date.today().isoformat(),
        'poligono': 'Av. Roosevelt - Buffer 100m',
        'area_m2': round(area_m2, 0),
        'area_ha': round(area_m2 / 10000, 2),
        'intersecciones_peatonales': stats['intersection_count'],
        'longitud_red_peatonal_km': round(stats['edge_length_total'] / 1000, 2),
        'longitud_promedio_segmento_m': round(stats['edge_length_avg'], 1),
        'densidad_calle_km_km2': round(densidad_km_km2, 2),
        'nodos_osm': len(G.nodes),
        'segmentos_osm': len(G.edges),
        'fuente': 'OpenStreetMap via OSMnx',
        'momento': 'linea_base_pre_intervencion'
    }
])

df_resultados.to_csv('../outputs/results/roosevelt_caminabilidad_linea_base.csv', index=False)
print('Resultados exportados a: outputs/results/roosevelt_caminabilidad_linea_base.csv')
print()
print(df_resultados.T.to_string())

In [ ]:
# Celda 8 — Descargar archivos generados (solo en Google Colab)
# from google.colab import files
# files.download('../outputs/results/roosevelt_caminabilidad_linea_base.csv')
# files.download('../outputs/figures/roosevelt_red_peatonal_linea_base.png')
print('Archivos generados en outputs/results/ y outputs/figures/')